# Pashto FastSpeech2: generate speech with a trained model (no training)

Loads a checkpoint saved to Google Drive by `colab_train.ipynb` and:
1. regenerates the training clips and compares them with the originals (overfitting test);
2. optionally speaks any Pashto sentence.

Runtime -> Change runtime type -> **GPU** (a CPU runtime also works, just slower).

## 1. Settings
`DRIVE_DIR` is the folder the training notebook used in its step 3. `CLIPS` must be the clips the
model was trained on. `STEP` is the checkpoint to load; `None` means the latest.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/pashto_tts"   # contains output/ckpt/Pashto/<step>.pth.tar
RUN_NAME  = "demo4"                               # results go to <DRIVE_DIR>/runs/<RUN_NAME>/
CLIPS     = '"1_crop-00[1-4]"'                     # the clips the model was trained on
STEP      = None                                  # e.g. 3000; None = latest checkpoint
STEP_ARG  = f"--step {STEP}" if STEP else ""

## 2. Get the repo
Clones this repo. FastSpeech2 is bundled inside it (`FastSpeech2/`), so nothing else is cloned. For a private repo, put a token in the URL or make the repo public.

In [ ]:
REPO_URL = "https://github.com/msds25033/pashto-fastspeech2.git"
import os
if not os.path.exists("/content/pashto-fastspeech2"):
    !git clone -q {REPO_URL} /content/pashto-fastspeech2
%cd /content/pashto-fastspeech2
!git log -1 --format="%h %s"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install (about 5 min, mostly the aligner)

In [ ]:
!pip install -q -r requirements.txt
!python pipeline.py setup
!python pipeline.py install-mfa

## 4. Connect the checkpoints on Google Drive
This links the repo's `FastSpeech2/output` and `runs` to the Drive folders. It only
deletes the empty local folders; nothing on Drive is removed.

In [ ]:
from google.colab import drive
import os, glob
drive.mount("/content/drive")
assert os.path.isdir(f"{DRIVE_DIR}/output/ckpt/Pashto"), f"no checkpoints in {DRIVE_DIR}/output/ckpt/Pashto"
!mkdir -p {DRIVE_DIR}/runs
!rm -rf FastSpeech2/output runs && ln -sfn {DRIVE_DIR}/output FastSpeech2/output && ln -sfn {DRIVE_DIR}/runs runs
print("checkpoints:", sorted(os.path.basename(p) for p in glob.glob("FastSpeech2/output/ckpt/Pashto/*.pth.tar")))

## 5. Prepare the training clips
The model needs the pitch/energy statistics and phone sequences of the clips it was
trained on. This recomputes them; it's deterministic, so the results match the training run.

In [ ]:
!python pipeline.py corpus --clips {CLIPS}
!python pipeline.py align
!python pipeline.py preprocess

## 6. Regenerate the training clips (overfitting test)
For every clip, six versions:
1. `original`: the recording.
2. `vocoder`: the real spectrogram played through HiFi-GAN; the best possible.
3. `teacher_forced`: the model given the real durations, pitch and energy.
4. `real_durations`: the model given only the real durations; it predicts pitch and energy.
5. `from_phones`: the model alone, from the training phones.
6. `from_text`: the model alone, from the Pashto text, with pauses at punctuation.

There are also spectrograms and metrics. In the metrics, lower `mel_l1_dtw` means closer to
the original, and a `length_ratio` near 1 means the same speaking rate. Comparing neighbours
shows where quality is lost:
- 3 vs 2: the decoder;
- 4 vs 3: the pitch/energy predictors;
- 5 vs 4: the duration predictor;
- 6 vs 5: the text front end.

In [ ]:
!python pipeline.py overfit-test --run-name {RUN_NAME} {STEP_ARG}

In [ ]:
import pandas as pd
from IPython.display import Image, Audio, display, Markdown
test_dir = sorted(glob.glob(f"runs/{RUN_NAME}/overfit_test/step*"), key=lambda d: int(d.rsplit("step", 1)[1]))[-1]
print(test_dir)
display(pd.read_csv(f"{test_dir}/metrics.csv"))
for png in sorted(glob.glob(f"{test_dir}/*_mels.png")):
    clip = os.path.basename(png)[:-len("_mels.png")]
    display(Markdown(f"### {clip}"))
    display(Image(png))
    for wav in sorted(glob.glob(f"{test_dir}/{clip}_[0-9]_*.wav")):
        print(os.path.basename(wav)); display(Audio(wav))

## 7. (Optional) Speak other Pashto sentences
Without arguments this reads `data/demo_sentences.txt`, which has one sentence from a training
clip and one unseen sentence. Or pass your own with `--text "..."`; repeat it for more.
Words missing from the lexicon are skipped with a warning.

In [ ]:
!python pipeline.py synthesize --run-name {RUN_NAME} {STEP_ARG}
# !python pipeline.py synthesize --run-name {RUN_NAME} {STEP_ARG} --text "دا یو مثال دی"
for wav in sorted(glob.glob(f"runs/{RUN_NAME}/audio/*.wav")):
    print(os.path.basename(wav)); display(Audio(wav))

## 8. Download the results
They are also saved on Drive, in `<DRIVE_DIR>/runs/<RUN_NAME>/`.

In [ ]:
!cd runs && zip -qr /content/{RUN_NAME}_generated.zip {RUN_NAME}/overfit_test {RUN_NAME}/audio
from google.colab import files
files.download(f"/content/{RUN_NAME}_generated.zip")